# Training the PICON

This notebook trains the PICON. 

In [27]:
import sys
import glob
import h5py
import torch as t
import numpy as np
from torch import nn
import multiprocessing
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from timm.models.layers import trunc_normal_

## Parameters

The neural network training result will be stored in "modelSaveDir", the learning rate is defined in "lRater". If using the physics-informed loss based on the PDE, then the "usePDELoss" should be True, and the weight of the PDE loss should be defined in "pdeWeight". 

In [28]:
modelSaveDir='OperatorSaverPIC3XFL/lr0.0003/'
lRater=0.0003
pdeWeight=1e-5
usePDELoss=False

timeGrid=0.005
gridNum=128
batchSmallSize=30#24
predLongTime=1
predLongStep=int(predLongTime/timeGrid)
muVal=5e-10

uD2k=t.meshgrid(*[t.fft.fftfreq(int(n)) for n in (gridNum,gridNum)],indexing="ij")
uD2k=t.stack(uD2k,-1)
uD2k=t.sum(uD2k**2,axis=-1).cuda()
fft_laplace=-((2*np.pi)**2)*uD2k
fft_laplace[0,0]=np.inf

## Functions

The unwrapDelta2 function uses the spectral method to calculate $\phi$ from $\Omega$.  
The periodicLaplace function calculates the $\nabla^2$ operator.  
The arakawaVec function calculates the poission bracket $[zeta,psi]$.  
The hwCalTD function calculates the temporal differentiation of $\Omega$ and $n$ using the numerical method the same as in the hw2d package.  
The rk4DiffCal function uses runge-kutta method to integrate the $\Omega$ and $n$ field over time.  
The LayerNorm, GRN, Block, ConvNeXtV2 functions define the PICON neural network, the input of the neural network is $\Omega$, $\phi$, $n$ as three-channel image, and time difference $t$, PDE parameters $c_1$, $k_0$, $\kappa$, $pbc$. The five scalar inputs are expanded to the 2-D images, so that the input of the neural network is a 8-channel image. The output of the neural network is a two-channel image, which is the $\Omega$, $n$ at $t$ after the input initial plasma state.  

In [29]:
def unwrapDelta2(tensor,dx):
    frequencies=t.fft.fft2(tensor)
    result=frequencies/(fft_laplace)
    result=t.where(fft_laplace==0,0,result)
    result=t.real(t.fft.ifft2(result))
    return result*dx**2
def periodicLaplace(fieldIn,dx):
    fieldPad=nn.functional.pad(fieldIn,(1,1,1,1),'circular')
    fieldOut=fieldPad[:,0:-2,1:-1]+fieldPad[:,1:-1,0:-2]-4*fieldPad[:,1:-1,1:-1]+fieldPad[:,1:-1,2:]+fieldPad[:,2:,1:-1]
    return fieldOut/dx**2
def arakawaVec(zeta,psi,dx):
    return (zeta[:,1:-1,2:  ]*(psi[:,2:  ,1:-1]-psi[:,0:-2,1:-1]+psi[:,2:  ,2:  ]-psi[:,0:-2,2:  ])
           -zeta[:,1:-1,0:-2]*(psi[:,2:  ,1:-1]-psi[:,0:-2,1:-1]+psi[:,2:  ,0:-2]-psi[:,0:-2,0:-2])
           -zeta[:,2:  ,1:-1]*(psi[:,1:-1,2:  ]-psi[:,1:-1,0:-2]+psi[:,2:  ,2:  ]-psi[:,2:  ,0:-2])
           +zeta[:,0:-2,1:-1]*(psi[:,1:-1,2:  ]-psi[:,1:-1,0:-2]+psi[:,0:-2,2:  ]-psi[:,0:-2,0:-2])
           +zeta[:,0:-2,2:  ]*(psi[:,1:-1,2:  ]-psi[:,0:-2,1:-1])
           +zeta[:,2:  ,2:  ]*(psi[:,2:  ,1:-1]-psi[:,1:-1,2:  ])
           -zeta[:,2:  ,0:-2]*(psi[:,2:  ,1:-1]-psi[:,1:-1,0:-2])
           -zeta[:,0:-2,0:-2]*(psi[:,1:-1,0:-2]-psi[:,0:-2,1:-1]))/(12*dx**2)
def hwCalTD(omegaDenIn,c1,kappa,pbc,spaGrid):
    phi1=unwrapDelta2(omegaDenIn[:,0],spaGrid)

    phiPad=nn.functional.pad(phi1,(1,1,1,1),'circular')
    denPad=nn.functional.pad(omegaDenIn[:,1],(1,1,1,1),'circular')
    phi1X=(phiPad[:,2:,1:-1]-phiPad[:,0:-2,1:-1])/(2*spaGrid)
    phi1Y=(phiPad[:,1:-1,2:]-phiPad[:,1:-1,0:-2])/(2*spaGrid)
    den1X=(denPad[:,2:,1:-1]-denPad[:,0:-2,1:-1])/(2*spaGrid)
    den1Y=(denPad[:,1:-1,2:]-denPad[:,1:-1,0:-2])/(2*spaGrid)
    omegaPad=nn.functional.pad(omegaDenIn[:,0],(1,1,1,1),'circular')
    omega1X=(omegaPad[:,2:,1:-1]-omegaPad[:,0:-2,1:-1])/(2*spaGrid)
    omega1Y=(omegaPad[:,1:-1,2:]-omegaPad[:,1:-1,0:-2])/(2*spaGrid)
    diffuTermDen=periodicLaplace(periodicLaplace(periodicLaplace(omegaDenIn[:,1],spaGrid),spaGrid),spaGrid)
    diffuTermOmg=periodicLaplace(periodicLaplace(periodicLaplace(omegaDenIn[:,0],spaGrid),spaGrid),spaGrid)
    denT1=c1.view([-1,1,1])*(phi1-omegaDenIn[:,1])-pbc.view([-1,1,1])*arakawaVec(phiPad,denPad,spaGrid)-kappa.view([-1,1,1])*phi1X+muVal*diffuTermDen
    omegaT1=c1.view([-1,1,1])*(phi1-omegaDenIn[:,1])-pbc.view([-1,1,1])*arakawaVec(phiPad,omegaPad,spaGrid)+muVal*diffuTermOmg
    
    omegaDenT=t.stack([omegaT1,denT1],axis=1)
    return omegaDenT

def rk4DiffCal(omegaDenIn,c1,kappa,pbc,spaGrid,tGrid):
    omegaDenT1=hwCalTD(omegaDenIn,c1,kappa,pbc,spaGrid)
    omegaDenIn2=omegaDenIn+omegaDenT1*tGrid.view([-1,1,1,1])/2
    omegaDenT2=hwCalTD(omegaDenIn2,c1,kappa,pbc,spaGrid)
    omegaDenIn3=omegaDenIn+omegaDenT2*tGrid.view([-1,1,1,1])/2
    omegaDenT3=hwCalTD(omegaDenIn3,c1,kappa,pbc,spaGrid)
    omegaDenIn4=omegaDenIn+omegaDenT3*tGrid.view([-1,1,1,1])
    omegaDenT4=hwCalTD(omegaDenIn4,c1,kappa,pbc,spaGrid)
    omegaDenDiff=(omegaDenT1+2*omegaDenT2+2*omegaDenT3+omegaDenT4)*tGrid.view([-1,1,1,1])/6
    return omegaDenDiff

class LayerNorm(nn.Module):
    """ LayerNorm that supports two data formats: channels_last (default) or channels_first. 
    The ordering of the dimensions in the inputs. channels_last corresponds to inputs with 
    shape (batch_size, height, width, channels) while channels_first corresponds to inputs 
    with shape (batch_size, channels, height, width).
    """
    def __init__(self,normalized_shape,eps=1e-6,data_format="channels_last"):
        super().__init__()
        self.weight=nn.Parameter(t.ones(normalized_shape))
        self.bias=nn.Parameter(t.zeros(normalized_shape))
        self.eps=eps
        self.data_format=data_format
        if self.data_format not in ["channels_last","channels_first"]:
            raise NotImplementedError 
        self.normalized_shape=(normalized_shape,)
    def forward(self,x):
        if self.data_format=="channels_last":
            return t.nn.functional.layer_norm(x,self.normalized_shape,self.weight,self.bias,self.eps)
        elif self.data_format == "channels_first":
            u=x.mean(1,keepdim=True)
            s=(x-u).pow(2).mean(1,keepdim=True)
            x=(x-u)/t.sqrt(s+self.eps)
            x=self.weight[:,None,None]*x+self.bias[:,None,None]
            return x

class GRN(nn.Module): #GRN (Global Response Normalization) layer
    def __init__(self,dim):
        super().__init__()
        self.gamma=nn.Parameter(t.zeros(1,1,1,dim))
        self.beta=nn.Parameter(t.zeros(1,1,1,dim))
    def forward(self,x):
        Gx=t.norm(x,p=2,dim=(1,2),keepdim=True)
        Nx=Gx/(Gx.mean(dim=-1,keepdim=True)+1e-6)
        return self.gamma*(x*Nx)+self.beta+x

class Block(nn.Module):
    """ ConvNeXtV2 Block.
    Args:
        dim (int): Number of input channels.
        drop_path (float): Stochastic depth rate. Default: 0.0
    """
    def __init__(self,dim,drop_path=0.0):
        super().__init__()
        self.dwconv=nn.Conv2d(dim,dim,kernel_size=7,padding=3,groups=dim,padding_mode='circular') # depthwise conv
        self.norm=LayerNorm(dim, eps=1e-6)
        self.pwconv1=nn.Linear(dim,4*dim) # pointwise/1x1 convs, implemented with linear layers
        self.act=nn.GELU()
        self.grn=GRN(4*dim)
        self.pwconv2=nn.Linear(4*dim,dim)
        self.drop_path=DropPath(drop_path) if drop_path>0.0 else nn.Identity()
    def forward(self,x):
        input=x
        x=self.dwconv(x)
        x=x.permute(0,2,3,1)# (N, C, H, W) -> (N, H, W, C)
        x=self.norm(x)
        x=self.pwconv1(x)
        x=self.act(x)
        x=self.grn(x)
        x=self.pwconv2(x)
        x=x.permute(0,3,1,2)# (N, H, W, C) -> (N, C, H, W)
        x=input+self.drop_path(x)
        return x

class ConvNeXtV2(nn.Module):
    """ ConvNeXt V2
    Args:
        in_chans (int): Number of input image channels. Default: 3
        num_classes (int): Number of classes for classification head. Default: 1000
        depths (tuple(int)): Number of blocks at each stage. Default: [3, 3, 9, 3]
        dims (int): Feature dimension at each stage. Default: [96, 192, 384, 768]
        drop_path_rate (float): Stochastic depth rate. Default: 0.
        head_init_scale (float): Init scaling value for classifier weights and biases. Default: 1.
    """
    def __init__(self,in_chans=8,out_channel=2,depths=[3,3,9,3],dims=[96,192,384,768],drop_path_rate=0.0,head_init_scale=1.0):
        super().__init__()
        self.depths=depths
        self.num_stage=len(depths)
        self.downsample_layers=nn.ModuleList() # stem and 3 intermediate downsampling conv layers
        stem = nn.Sequential(nn.Conv2d(in_chans,dims[0],kernel_size=2,stride=2),
                             LayerNorm(dims[0],eps=1e-6,data_format="channels_first"))
        self.downsample_layers.append(stem)
        for i in range(self.num_stage-1):
            downsample_layer=nn.Sequential(LayerNorm(dims[i],eps=1e-6,data_format="channels_first"),
                                           nn.Conv2d(dims[i],dims[i+1],kernel_size=2,stride=2))
            self.downsample_layers.append(downsample_layer)
        self.stages=nn.ModuleList() # 4 feature resolution stages, each consisting of multiple residual blocks
        dp_rates=[x.item() for x in t.linspace(0,drop_path_rate,sum(depths))] 
        cur=0
        for i in range(self.num_stage):
            stage=nn.Sequential(*[Block(dim=dims[i],drop_path=dp_rates[cur+j]) for j in range(depths[i])])
            self.stages.append(stage)
            cur+=depths[i]
        self.norm=nn.LayerNorm(dims[-1],eps=1e-6)# final norm layer
        self.out_conv=nn.Conv2d(int(dims[0]/2),out_channel,kernel_size=1) # final classifier conv
        self.upsample_layers=nn.ModuleList() # stem and 3 intermediate downsampling conv layers
        for i in reversed(range(self.num_stage)):
            upsample_layer = nn.Sequential(LayerNorm(dims[i]*2,eps=1e-6,data_format="channels_first"),
                                           nn.ConvTranspose2d(dims[i]*2,int(dims[i]/2),kernel_size=2,stride=2))
            self.upsample_layers.append(upsample_layer)
        self.apply(self._init_weights)
    def _init_weights(self,m):
        if isinstance(m,(nn.Conv2d,nn.Linear)):
            trunc_normal_(m.weight,std=0.02)
            nn.init.constant_(m.bias,0)
    def Encoder(self,x):
        down_features=[]
        for i in range(self.num_stage):
            x=self.downsample_layers[i](x)
            x=self.stages[i](x)
            down_features.append(x)
        return x,down_features
    def Decoder(self,x,down_features):
        for i in range(self.num_stage):
            x=t.cat([x,down_features.pop()],dim=1)
            x=self.upsample_layers[i](x)
        return x
    def forward(self,x):
        x,down_features=self.Encoder(x)
        x=self.Decoder(x, down_features)
        #x=self.norm(x)#Gesa: The copy code forgot to use this norm layer, and the norm layer seems in wrong shape
        x=self.out_conv(x)
        return x

## The Neural Network Initialized

In [30]:
PICNXnet=ConvNeXtV2().float().cuda()
optimPICNX=t.optim.AdamW(PICNXnet.parameters(),lRater)

In [31]:
chanRescale=t.tensor([100,20]).reshape([1,2,1,1]).float().cuda()

## Training the Neural Network

The neural network will read the data in batches and trained. Note that the inclusion of PDE will increase the training time by a factor of 3, while the training accuracy is not significantly increased. An epoch of training takes several hours using a NVIDIA A100 GPU, and the paper only finished about 16 epochs of training. 

In [32]:
lossList=[]
mseLossList=[]
pdeLossList=[]

for epoch in range(2000):
    for horiInd in range(100):
        omegaPhiDenInList=t.tensor(np.load('TDHiData/omegaPhiDenInList_'+str(horiInd)+'.npy')).float()
        omegaDenOutList=t.tensor(np.load('TDHiData/omegaPhiDenOutList_'+str(horiInd)+'.npy')[:,[0,2]]).float()
        timeStepList=t.tensor(np.load('TDHiData/tDifferList_'+str(horiInd)+'.npy')).float()
        c1List=t.tensor(np.load('TDHiData/c1List_'+str(horiInd)+'.npy')).float()
        k0List=t.tensor(np.load('TDHiData/k0List_'+str(horiInd)+'.npy')).float()
        kappaList=t.tensor(np.load('TDHiData/kappaList_'+str(horiInd)+'.npy')).float()
        pbcList=t.tensor(np.load('TDHiData/pbcList_'+str(horiInd)+'.npy')).float()
        
        shuArr=np.arange(len(omegaPhiDenInList))
        np.random.shuffle(shuArr)

        omegaPhiDenInList=omegaPhiDenInList[shuArr].split(batchSmallSize)
        omegaDenOutList=omegaDenOutList[shuArr].split(batchSmallSize)
        timeStepList=timeStepList[shuArr].split(batchSmallSize)
        c1List=c1List[shuArr].split(batchSmallSize)
        k0List=k0List[shuArr].split(batchSmallSize)
        kappaList=kappaList[shuArr].split(batchSmallSize)
        pbcList=pbcList[shuArr].split(batchSmallSize)
        
        for batch in range(len(omegaPhiDenInList)):
            omegaPhiDenIn=omegaPhiDenInList[batch].cuda()
            omegaDenOut=omegaDenOutList[batch].cuda()
            timeStep=timeStepList[batch].reshape([-1,1]).cuda()
            c1=c1List[batch].reshape([-1,1,1,1]).cuda()
            k0=k0List[batch].reshape([-1,1,1,1]).cuda()
            kappa=kappaList[batch].reshape([-1,1,1,1]).cuda()
            pbc=pbcList[batch].reshape([-1,1,1,1]).cuda()
            
            LBox=2*np.pi/k0
            spaGrid=(LBox/gridNum).reshape([-1,1,1])#Note that the simulation is in 256x256, while the stored data is downscaled to 128x128, checked, no change required. 
            
            PICNXnet.zero_grad()
            optimPICNX.zero_grad()
            if usePDELoss:
                def jacIn(tDiffIn):
                    OPDTCKKP=t.cat([omegaPhiDenIn,
                                    tDiffIn.reshape([-1,1,1,1])*t.ones([1,1,128,128]).cuda(),
                                    c1*t.ones([1,1,128,128]).cuda(),
                                    k0*t.ones([1,1,128,128]).cuda(),
                                    kappa*t.ones([1,1,128,128]).cuda(),
                                    pbc*t.ones([1,1,128,128]).cuda()
                                   ],axis=1)
                    Encoded,midStage=PICNXnet.Encoder(OPDTCKKP)
                    omegaDenPred=PICNXnet.out_conv(PICNXnet.Decoder(Encoded,midStage))*tDiffIn.reshape([-1,1,1,1])*chanRescale+omegaPhiDenIn[:,[0,2]]
                    return omegaDenPred
                omegaDenPred,omegaDenT=t.autograd.functional.jvp(jacIn,timeStep,v=t.ones([len(timeStep),1]).cuda(),create_graph=True)
                
                mseLoss=((omegaDenPred-omegaDenOut)/chanRescale)**2
                mseLoss=t.mean(mseLoss)
                
                omegaPred=omegaDenPred[:,0]
                phiPred=unwrapDelta2(omegaPred,spaGrid)
                denPred=omegaDenPred[:,1]
                phiPad=nn.functional.pad(phiPred,(1,1,1,1),'circular')
                denPad=nn.functional.pad(denPred,(1,1,1,1),'circular')
                phiX=(phiPad[:,2:,1:-1]-phiPad[:,0:-2,1:-1])/(2*spaGrid)
                omegaPad=nn.functional.pad(omegaPred,(1,1,1,1),'circular')
                diffuTermDen=periodicLaplace(periodicLaplace(periodicLaplace(denPred,spaGrid),spaGrid),spaGrid)
                diffuTermOmg=periodicLaplace(periodicLaplace(periodicLaplace(omegaPred,spaGrid),spaGrid),spaGrid)
    
                pde1Right=c1.view([-1,1,1])*(phiPred-denPred)-pbc.view([-1,1,1])*arakawaVec(phiPad,denPad,spaGrid)-kappa.view([-1,1,1])*phiX+muVal*diffuTermDen
                pde2Right=c1.view([-1,1,1])*(phiPred-denPred)-pbc.view([-1,1,1])*arakawaVec(phiPad,omegaPad,spaGrid)+muVal*diffuTermOmg
                pdeLoss1=t.mean((pde1Right-omegaDenT[:,1])**2)
                pdeLoss2=t.mean((pde2Right-omegaDenT[:,0])**2)
                pdeLoss=pdeLoss1*0.005+pdeLoss2*0.00005
                loss=mseLoss+pdeLoss*pdeWeight
                loss.backward()
                optimPICNX.step()
                
                print(batch,loss.item(),pdeLoss1.item(),pdeLoss2.item(),mseLoss.item())
                lossList.append(loss.item())
                pdeLossList.append(pdeLoss.item())
                mseLossList.append(mseLoss.item())
                assert np.isnan(loss.item())==False
            else:
                OPDTCKKP=t.cat([omegaPhiDenIn,
                            timeStep.reshape([-1,1,1,1])*t.ones([1,1,128,128]).cuda(),
                            c1*t.ones([1,1,128,128]).cuda(),
                            k0*t.ones([1,1,128,128]).cuda(),
                            kappa*t.ones([1,1,128,128]).cuda(),
                            pbc*t.ones([1,1,128,128]).cuda()
                           ],axis=1)
                Encoded,midStage=PICNXnet.Encoder(OPDTCKKP)
                omegaDenPred=PICNXnet.out_conv(PICNXnet.Decoder(Encoded,midStage))*timeStep.reshape([-1,1,1,1])*chanRescale+omegaPhiDenIn[:,[0,2]]
                mseLoss=((omegaDenPred-omegaDenOut)/chanRescale)**2
                mseLoss=t.mean(mseLoss)
                loss=mseLoss
                loss.backward()
                optimPICNX.step()
                
                print(batch,loss.item())
                lossList.append(loss.item())
            
        t.save(PICNXnet,modelSaveDir+'PICNX.pt')
        np.save(modelSaveDir+'lossList.npy',np.array(lossList))
        np.save(modelSaveDir+'pdeLossList.npy',np.array(pdeLossList))
        np.save(modelSaveDir+'mseLossList.npy',np.array(mseLossList))

FileNotFoundError: [Errno 2] No such file or directory: 'TDHiData/omegaPhiDenInList_0.npy'